In [1]:
# Import all the required libraries
import pandas as pd
import numpy as np
# import matplotlib.pyplot as plt
# from matplotlib.colors import LogNorm
import time
import gc
from scipy.stats import pearsonr
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, mean_squared_error, mean_absolute_error, r2_score

In [2]:
df_initial = pd.DataFrame()

In [3]:
# Defining column names because the original data file didn't have column headers
column_names = ['ii', 'jj', 'iyr', 'imo', 'idy', 'ihr', 'imn', 'isc',
                'zlat1', 'zlon1', 'ztcs11', 'ztcs12', 'ztcs13', 'ztcs14', 'ztcs15', 'ztcs16', 'ztcs17', 'ztcs18', 'ztcs19',
                'iiG', 'jjG', 'zGdmin', 'iyrG', 'imoG', 'idyG', 'ihrG', 'imnG', 'iscG',
                'zlat2', 'zlon2', 'ztcs21', 'ztcs22', 'ztcs23', 'ztcs24',
                'iiD', 'jjD', 'zDdmin', 'jyrD', 'jmoD', 'jdyD', 'jhrD', 'jmnD', 'jscD',
                'zlatFS', 'zlonFS', 'zDPR1', 'zDPR2', 'zDPR3',
                'zlat', 'zlon', 'zpcp', 'ipop', 'isfc', 'itemp', 'itcw', 'IYNo']

In [4]:
# # Define a function to read the file line by line
# def read_large_file(file_path):
#     with open(file_path, 'r') as file:
#         for line in file:
#             yield line.strip()

In [5]:
# # Stream the data line-by-line
# for i, line in enumerate(read_large_file('match_GMI-GPF-DPR_2015_5pos.txt')):
    
#     row_data = pd.DataFrame([line.split()]) # Split elements of row with any amount of whitespace between them
    
#     df_initial = pd.concat([df_initial, row_data], ignore_index=True) # Adding data to the new dataframe
    
#     if i == 10: # Stopping at a particular # of rows
#         break

In [6]:
# Assigning column names
# df_initial.columns = column_names

#### So this indicates that we have more than 24k -ve values and 34k -ve values for zDPR2 and zDPR3 respectively (no -ve value for zDPR1)

In [7]:
# Measure how much time it takes to read in allvalues from all 3 columns
start_time = time.time()

df_initial = pd.read_csv('match_GMI-GPF-DPR_2015_5pos.txt', header = None, delim_whitespace = True)

# Mark the end of code execution
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Execution time: {elapsed_time:.5f} seconds")

/tmp/ipykernel_121772/597419636.py:4: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df_initial = pd.read_csv('match_GMI-GPF-DPR_2015_5pos.txt', header = None, delim_whitespace = True)


Execution time: 391.67191 seconds


In [8]:
# Assigning column names
df_initial.columns = column_names

In [9]:
rainrate_label = 'zDPR1'

In [10]:
# Filtering out elements with faulty values: -99.99 for DPR rainrates and -1 for Yes/No flag
df = df_initial[(df_initial['zDPR2'] >= 0) & (df_initial['zDPR3'] >= 0) & (df_initial['IYNo'] >= 0)]

In [11]:
# Printing the df
df

,ii,jj,iyr,imo,idy,ihr,imn,isc,zlat1,zlon1,...,zDPR2,zDPR3,zlat,zlon,zpcp,ipop,isfc,itemp,itcw,IYNo
0,81,109,2015,1,1,1,32,3,-64.496,-106.303,...,0.0,0.00,-64.496,-106.303,0.05,15,1,274,7,1
1,82,109,2015,1,1,1,32,5,-64.472,-106.035,...,0.0,0.00,-64.472,-106.035,0.04,14,1,274,8,1
2,82,110,2015,1,1,1,32,5,-64.524,-106.012,...,0.0,0.00,-64.524,-106.012,0.05,17,1,274,8,1
3,83,109,2015,1,1,1,32,7,-64.447,-105.768,...,0.0,0.00,-64.447,-105.768,0.05,17,1,274,8,1
4,83,110,2015,1,1,1,32,7,-64.499,-105.744,...,0.0,0.00,-64.499,-105.744,0.05,17,1,274,8,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73929474,2961,109,2016,1,1,0,20,35,-64.663,87.489,...,0.0,0.00,-64.663,87.489,0.01,6,1,272,5,0
73929475,2961,110,2016,1,1,0,20,35,-64.716,87.467,...,0.0,0.02,-64.716,87.467,0.02,8,1,272,5,0
73929476,2961,111,2016,1,1,0,20,35,-64.768,87.446,...,0.0,0.02,-64.768,87.446,0.03,11,1,272,5,0
73929477,2961,112,2016,1,1,0,20,35,-64.820,87.427,...,0.0,0.02,-64.820,87.427,0.03,11,1,272,5,0


In [12]:
# Subsetting data into ocean and land
df_ocean_initial = df[df['isfc'] == 1]
df_land_initial = df[(df['isfc'] >= 3) & (df['isfc'] <= 7)]

In [13]:
df_ocean_initial

,ii,jj,iyr,imo,idy,ihr,imn,isc,zlat1,zlon1,...,zDPR2,zDPR3,zlat,zlon,zpcp,ipop,isfc,itemp,itcw,IYNo
0,81,109,2015,1,1,1,32,3,-64.496,-106.303,...,0.0,0.00,-64.496,-106.303,0.05,15,1,274,7,1
1,82,109,2015,1,1,1,32,5,-64.472,-106.035,...,0.0,0.00,-64.472,-106.035,0.04,14,1,274,8,1
2,82,110,2015,1,1,1,32,5,-64.524,-106.012,...,0.0,0.00,-64.524,-106.012,0.05,17,1,274,8,1
3,83,109,2015,1,1,1,32,7,-64.447,-105.768,...,0.0,0.00,-64.447,-105.768,0.05,17,1,274,8,1
4,83,110,2015,1,1,1,32,7,-64.499,-105.744,...,0.0,0.00,-64.499,-105.744,0.05,17,1,274,8,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73929474,2961,109,2016,1,1,0,20,35,-64.663,87.489,...,0.0,0.00,-64.663,87.489,0.01,6,1,272,5,0
73929475,2961,110,2016,1,1,0,20,35,-64.716,87.467,...,0.0,0.02,-64.716,87.467,0.02,8,1,272,5,0
73929476,2961,111,2016,1,1,0,20,35,-64.768,87.446,...,0.0,0.02,-64.768,87.446,0.03,11,1,272,5,0
73929477,2961,112,2016,1,1,0,20,35,-64.820,87.427,...,0.0,0.02,-64.820,87.427,0.03,11,1,272,5,0


In [14]:
del df_initial, df  # Delete the DataFrame
gc.collect()  # Run garbage collection to free memory

14

In [15]:
# # Counting no of values between 0 and 0.01
# x = 0  # lower bound
# y = 0.01  # upper bound

# count = ((df_ocean_initial[rainrate_label] >= x) & (df_ocean_initial[rainrate_label] <= y)).sum()
# print(count)

#### Using 0.01 mm/h as a threshold, we classify rows into raining(1)/non-raining(0)

In [16]:
rainrate_threshold = 0.01

In [17]:
binary_column_for_rain_norain = 'IYNo_Radar'

## Part 1: Ocean Data

In [18]:
df_ocean_initial.loc[:, binary_column_for_rain_norain] = np.where(df_ocean_initial[rainrate_label] >= rainrate_threshold, 1, 0)

/tmp/ipykernel_121772/2478116695.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ocean_initial.loc[:, binary_column_for_rain_norain] = np.where(df_ocean_initial[rainrate_label] >= rainrate_threshold, 1, 0)


In [19]:
counts = df_ocean_initial[binary_column_for_rain_norain].value_counts()
count_of_ones = counts[1]

In [20]:
count_of_initial_rows_to_delete = len(df_ocean_initial) - (2 * counts[1])
count_of_initial_rows_to_delete

40529322

In [21]:
# Step 1: Get all rows where the condition is met
rows_to_delete = df_ocean_initial[df_ocean_initial[binary_column_for_rain_norain] == 0]

# Step 2: Remove the first `x` rows from the filtered rows
df_ocean_initial = df_ocean_initial.drop(rows_to_delete.iloc[:count_of_initial_rows_to_delete].index)

In [22]:
df_ocean = df_ocean_initial.sample(frac = 0.01, random_state = 38)

In [174]:
# Printing the df
df_ocean

68997

### 1.1.1) Ocean Data, "ocean_ATMS_LF+HF", Random Forest, Step 1, Classification (Rain/No-Rain)

In [24]:
# Run for "ocean_ATMS_LF+HF"
X = df_ocean[['ztcs15', 'ztcs16', 'ztcs18', 'ztcs21', 'ztcs23', 'ztcs24']]

In [25]:
# # Printing the df
# X

In [26]:
y = df_ocean[binary_column_for_rain_norain]

In [27]:
# Split data into training and validation sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 16)

In [28]:
# Measure how much time it takes to read in allvalues from all 3 columns
start_time = time.time()

rf = RandomForestClassifier(n_estimators = 200, random_state = 50)

# Train the model
rf.fit(X_train, y_train)

# Mark the end of code execution
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Execution time: {elapsed_time:.5f} seconds")

Execution time: 15.70842 seconds


In [29]:
# Make predictions on the test set
y_test_pred = rf.predict(X_test)

In [30]:
# Calculating metrics
accuracy = accuracy_score(y_test, y_test_pred)
conf_matrix = confusion_matrix(y_test, y_test_pred)
class_report = classification_report(y_test, y_test_pred)

In [31]:
# Expressing the confusion matrix in terms of ratios (normalized)
conf_matrix_normalized = np.round(conf_matrix.astype('float') / conf_matrix.sum(axis=1)[:, np.newaxis] * 100, 2)

In [32]:
# Print evaluation metrics
print(f'Accuracy: {accuracy * 100:.2f}%')
print('\nArrangement of the confusion matrix below: \n [[TN \t FP] \n [FN \t TP]]')
print('\nConfusion Matrix:')
print(conf_matrix_normalized)
print('\nClassification Report:')
print(class_report)

Accuracy: 89.97%

Arrangement of the confusion matrix below: 
 [[TN 	 FP] 
 [FN 	 TP]]

Confusion Matrix:
[[89.09 10.91]
 [ 9.15 90.85]]

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.89      0.90      6913
           1       0.89      0.91      0.90      6887

    accuracy                           0.90     13800
   macro avg       0.90      0.90      0.90     13800
weighted avg       0.90      0.90      0.90     13800



In [33]:
# Getting indices of X_val
index_numbers = X_test.index.tolist()

In [34]:
# Create the DataFrame with only entries where Predictions is 1
classification_prediction_df = pd.DataFrame({
    'Index': [index for i, index in enumerate(index_numbers) if y_test_pred[i] == 1],
    'Predictions': [pred for pred in y_test_pred if pred == 1]
})

In [35]:
# classification_prediction_df = classification_prediction_df.sort_values(by = 'Index').reset_index(drop = False)
classification_prediction_df = classification_prediction_df.sort_values(by = 'Index').reset_index(drop = True)

In [36]:
# # Printing the df
# classification_prediction_df

### 1.1.2) Ocean Data, "ocean_ATMS_LF+HF", Random Forest, Step 2, Regression (Amount of Rain)

In [37]:
df_ocean_initial_above_threshold = df_ocean_initial[(df_ocean_initial[rainrate_label] >= rainrate_threshold)]

In [38]:
# Filter out rows where the index matches indices_to_delete
df_ocean_initial_above_threshold_training_data = df_ocean_initial_above_threshold.drop(index = classification_prediction_df['Index'][classification_prediction_df['Index'].isin(df_ocean_initial_above_threshold.index)])
# df_ocean_initial_above_threshold_training_data

In [39]:
df_ocean_initial_above_threshold_training_data_sampled = df_ocean_initial_above_threshold_training_data.sample(frac = 0.05, random_state = 38)
df_ocean_initial_above_threshold_training_data_sampled

,ii,jj,iyr,imo,idy,ihr,imn,isc,zlat1,zlon1,...,zDPR3,zlat,zlon,zpcp,ipop,isfc,itemp,itcw,IYNo,IYNo_Radar
43997506,2621,111,2015,7,19,2,34,52,-46.396,11.260,...,0.04,-46.396,11.260,0.13,23,1,276,5,0,1
10006105,763,112,2015,2,15,23,9,0,-1.382,164.875,...,0.50,-1.382,164.875,0.62,88,1,301,60,1,1
12296278,2948,113,2015,2,26,2,27,20,-64.509,-33.311,...,0.20,-64.509,-33.311,0.03,12,1,269,4,0,1
15087275,1564,113,2015,3,10,9,57,47,64.442,7.943,...,0.40,64.442,7.943,0.02,8,1,279,12,0,1
1487971,1565,112,2015,1,8,7,1,29,61.512,-21.387,...,0.23,61.512,-21.387,0.24,42,1,276,5,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71397755,686,112,2015,12,20,9,30,49,-2.114,100.290,...,0.67,-2.114,100.290,0.29,65,1,301,57,1,1
64452247,1078,109,2015,11,18,17,15,26,32.992,141.537,...,2.60,32.992,141.537,3.42,100,1,296,53,1,1
54316186,751,113,2015,10,4,0,16,45,5.059,-138.410,...,2.98,5.059,-138.410,3.21,100,1,301,59,1,1
27739200,1863,112,2015,5,6,22,33,35,42.400,-21.694,...,0.13,42.400,-21.694,0.05,31,1,288,27,1,1


In [40]:
# Run for "ocean_ATMS_LF+HF"
X_train = df_ocean_initial_above_threshold_training_data_sampled[['ztcs15', 'ztcs16', 'ztcs18', 'ztcs21', 'ztcs23', 'ztcs24']]

In [41]:
X_train

,ztcs15,ztcs16,ztcs18,ztcs21,ztcs23,ztcs24
43997506,193.04,207.27,236.86,256.36,259.08,262.52
10006105,257.68,247.87,276.81,268.26,249.93,259.93
12296278,189.91,204.23,233.94,246.67,249.18,253.32
15087275,199.67,208.48,245.11,258.89,247.50,255.50
1487971,194.35,206.45,234.50,251.86,248.33,254.71
...,...,...,...,...,...,...
71397755,254.58,241.02,280.46,276.29,254.67,265.44
64452247,257.67,247.30,210.66,176.78,209.06,188.03
54316186,266.31,267.71,266.68,252.33,251.05,251.16
27739200,220.74,220.09,262.63,267.66,251.31,259.31


In [42]:
y_train = df_ocean_initial_above_threshold_training_data_sampled[rainrate_label]

In [43]:
# Convert labels to a logarithmic scale
y_train_log = np.log(y_train)

In [44]:
# Split data into training and validation sets
X_train, X_val, y_train_log, y_val_log = train_test_split(X_train, y_train_log, test_size = 0.2, random_state = 16)

In [45]:
rf = RandomForestRegressor(n_estimators = 100, random_state = 50)

# Train the model
rf.fit(X_train, y_train_log)

RandomForestRegressor(random_state=50)

In [46]:
# Make predictions on the validation set
y_val_pred_log = rf.predict(X_val)

In [47]:
y_val_pred = np.exp(y_val_pred_log)
y_val_pred = np.exp(y_val_pred_log)

In [48]:
y_val = np.exp(y_val_log)

In [49]:
# Calculate R-squared
r2 = r2_score(y_val, y_val_pred)
print("R-squared:", r2)

# Calculate Mean Squared Error
mse = mean_squared_error(y_val, y_val_pred)
rmse = np.sqrt(mse)
print("Root Mean Squared Error:", rmse)

# Calculate Mean Absolute Error
mae = mean_absolute_error(y_val, y_val_pred)
print("Mean Absolute Error:", mae)

correlation_coefficient = pearsonr(y_val, y_val_pred)[0]
print(f'Correlation Coefficient: {correlation_coefficient:.2f}')

R-squared: 0.16597083294850834
Root Mean Squared Error: 6.064662989029764
Mean Absolute Error: 0.9589613793107599
Correlation Coefficient: 0.45


In [50]:
indices = classification_prediction_df['Index']

In [51]:
selected_rows = df_ocean_initial.loc[indices]

In [52]:
selected_rows

,ii,jj,iyr,imo,idy,ihr,imn,isc,zlat1,zlon1,...,zDPR3,zlat,zlon,zpcp,ipop,isfc,itemp,itcw,IYNo,IYNo_Radar
26808,244,110,2015,1,1,4,42,13,-55.177,-118.921,...,1.27,-55.177,-118.921,1.01,96,1,280,19,1,1
32472,1377,109,2015,1,1,5,17,37,60.340,-31.919,...,0.25,60.340,-31.919,0.21,29,1,276,6,1,1
39114,98,113,2015,1,1,6,10,11,-64.218,-171.957,...,0.62,-64.218,-171.957,0.66,89,1,275,24,1,1
40124,300,113,2015,1,1,6,16,30,-50.410,-134.736,...,1.93,-50.410,-134.736,0.32,81,1,282,14,1,1
40184,312,113,2015,1,1,6,16,52,-49.301,-133.358,...,3.53,-49.301,-133.358,4.31,100,1,282,16,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73909862,1600,113,2015,12,31,22,5,31,63.212,-25.185,...,0.09,63.212,-25.185,0.16,26,1,275,6,1,0
73915514,2733,113,2015,12,31,22,40,55,-50.386,66.963,...,0.40,-50.386,66.963,1.28,99,1,278,21,1,1
73915624,2755,113,2015,12,31,22,41,36,-52.357,69.653,...,0.01,-52.357,69.653,0.96,94,1,275,18,1,0
73915722,2775,111,2015,12,31,22,42,14,-54.008,72.446,...,0.10,-54.008,72.446,0.30,49,1,274,13,1,0


In [53]:
# Run for "ocean_ATMS_LF+HF"
X_test = selected_rows[['ztcs15', 'ztcs16', 'ztcs18', 'ztcs21', 'ztcs23', 'ztcs24']]

In [54]:
X_test

,ztcs15,ztcs16,ztcs18,ztcs21,ztcs23,ztcs24
26808,215.57,225.51,253.62,246.90,245.47,248.06
32472,199.14,211.76,244.04,259.28,249.88,257.89
39114,228.92,239.90,267.18,266.11,253.58,261.45
40124,212.18,225.91,261.12,267.58,256.54,263.47
40184,235.72,252.58,261.04,257.44,255.31,259.16
...,...,...,...,...,...,...
73909862,194.57,207.82,236.69,246.56,244.12,248.59
73915514,232.62,245.58,268.86,265.77,248.28,259.75
73915624,224.63,239.07,266.61,266.25,249.67,260.81
73915722,200.86,206.56,235.26,237.51,245.52,245.48


In [55]:
y_test = selected_rows[rainrate_label]

In [56]:
y_test

26808       1.04
32472       1.57
39114       0.38
40124       1.85
40184       5.13
            ... 
73909862    0.00
73915514    1.02
73915624    0.00
73915722    0.00
73920023    0.00
Name: zDPR1, Length: 7011, dtype: float64

In [57]:
# Make predictions on the test dataset
y_test_pred_log = rf.predict(X_test)

In [58]:
y_test_pred_log

array([-0.28811972, -0.86762237, -0.47915495, ..., -0.40066255,
       -0.80850981, -0.41711447])

In [59]:
# Applying anti-log over the results
y_test_pred = np.exp(y_test_pred_log)

In [60]:
# Calculate R-squared
r2 = r2_score(y_test, y_test_pred)
print(f'R-squared: {r2:.2f}')

# Calculate Mean Squared Error
mse = mean_squared_error(y_test, y_test_pred)
rmse = np.sqrt(mse)
print(f'Root Mean Squared Error: {rmse:.2f}')

# Calculate Mean Absolute Error
mae = mean_absolute_error(y_test, y_test_pred)
print(f'Mean Absolute Error: {mae:.2f}')

correlation_coefficient = pearsonr(y_test, y_test_pred)[0]
print(f'Correlation Coefficient: {correlation_coefficient:.2f}')

R-squared: 0.18
Root Mean Squared Error: 5.22
Mean Absolute Error: 0.95
Correlation Coefficient: 0.44


### 1.2.1) Ocean Data, "ocean_ATMS_HF", Random Forest, Step 1, Classification (Rain/No-Rain)

In [61]:
# Run for "ocean_ATMS_HF"
X = df_ocean[['ztcs18', 'ztcs21', 'ztcs23', 'ztcs24']]

In [62]:
# # Printing the df
# X

In [63]:
y = df_ocean[binary_column_for_rain_norain]

In [64]:
# Split data into training and validation sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 16)

In [65]:
# Measure how much time it takes to read in allvalues from all 3 columns
start_time = time.time()

rf = RandomForestClassifier(n_estimators = 200, random_state = 50)

# Train the model
rf.fit(X_train, y_train)

# Mark the end of code execution
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Execution time: {elapsed_time:.5f} seconds")

Execution time: 15.18517 seconds


In [66]:
# Make predictions on the test set
y_test_pred = rf.predict(X_test)

In [67]:
# Calculating metrics
accuracy = accuracy_score(y_test, y_test_pred)
conf_matrix = confusion_matrix(y_test, y_test_pred)
class_report = classification_report(y_test, y_test_pred)

In [68]:
# Expressing the confusion matrix in terms of ratios (normalized)
conf_matrix_normalized = np.round(conf_matrix.astype('float') / conf_matrix.sum(axis=1)[:, np.newaxis] * 100, 2)

In [69]:
# Print evaluation metrics
print(f'Accuracy: {accuracy * 100:.2f}%')
print('\nArrangement of the confusion matrix below: \n [[TN \t FP] \n [FN \t TP]]')
print('\nConfusion Matrix:')
print(conf_matrix_normalized)
print('\nClassification Report:')
print(class_report)

Accuracy: 85.83%

Arrangement of the confusion matrix below: 
 [[TN 	 FP] 
 [FN 	 TP]]

Confusion Matrix:
[[85.22 14.78]
 [13.55 86.45]]

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.85      0.86      6913
           1       0.85      0.86      0.86      6887

    accuracy                           0.86     13800
   macro avg       0.86      0.86      0.86     13800
weighted avg       0.86      0.86      0.86     13800



In [70]:
# Getting indices of X_val
index_numbers = X_test.index.tolist()

In [71]:
# Create the DataFrame with only entries where Predictions is 1
classification_prediction_df = pd.DataFrame({
    'Index': [index for i, index in enumerate(index_numbers) if y_test_pred[i] == 1],
    'Predictions': [pred for pred in y_test_pred if pred == 1]
})

In [72]:
# classification_prediction_df = classification_prediction_df.sort_values(by = 'Index').reset_index(drop = False)
classification_prediction_df = classification_prediction_df.sort_values(by = 'Index').reset_index(drop = True)

In [73]:
# # Printing the df
# classification_prediction_df

### 1.2.2) Ocean Data, "ocean_ATMS_HF", Random Forest, Step 2, Regression (Amount of Rain)

In [74]:
df_ocean_initial_above_threshold = df_ocean_initial[(df_ocean_initial[rainrate_label] >= rainrate_threshold)]

In [75]:
# Filter out rows where the index matches indices_to_delete
df_ocean_initial_above_threshold_training_data = df_ocean_initial_above_threshold.drop(index = classification_prediction_df['Index'][classification_prediction_df['Index'].isin(df_ocean_initial_above_threshold.index)])
# df_ocean_initial_above_threshold_training_data

In [76]:
df_ocean_initial_above_threshold_training_data_sampled = df_ocean_initial_above_threshold_training_data.sample(frac = 0.05, random_state = 38)
df_ocean_initial_above_threshold_training_data_sampled

,ii,jj,iyr,imo,idy,ihr,imn,isc,zlat1,zlon1,...,zDPR3,zlat,zlon,zpcp,ipop,isfc,itemp,itcw,IYNo,IYNo_Radar
16919479,182,110,2015,3,18,13,8,15,-59.831,128.443,...,0.55,-59.831,128.443,0.52,72,1,275,12,1,1
9884188,2665,109,2015,2,15,10,15,24,-43.699,-152.325,...,0.44,-43.699,-152.325,0.63,100,1,291,39,1,1
70558267,2227,109,2015,12,16,16,50,21,-4.443,-170.400,...,0.74,-4.443,-170.400,0.88,97,1,301,65,1,1
32872149,2687,110,2015,5,29,23,18,32,-45.809,-80.269,...,0.70,-45.809,-80.269,0.33,75,1,284,19,1,1
14002835,862,112,2015,3,5,15,26,7,9.525,-151.853,...,0.95,9.525,-151.853,2.03,100,1,299,58,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29948179,107,113,2015,5,16,23,52,2,-63.929,49.250,...,0.53,-63.929,49.250,0.03,9,1,272,8,0,1
27055259,2276,111,2015,5,3,11,30,48,-9.970,-171.029,...,0.48,-9.970,-171.029,0.81,95,1,300,61,1,1
44495506,1563,111,2015,7,21,8,2,13,61.552,-168.259,...,1.46,61.552,-168.259,1.10,100,1,282,24,1,1
50077389,547,110,2015,8,15,21,32,15,-25.129,103.305,...,4.64,-25.129,103.305,2.48,100,1,292,35,1,1


In [77]:
# Run for "ocean_ATMS_HF"
X_train = df_ocean_initial_above_threshold_training_data_sampled[['ztcs18', 'ztcs21', 'ztcs23', 'ztcs24']]

In [78]:
X_train

,ztcs18,ztcs21,ztcs23,ztcs24
16919479,235.73,236.48,246.80,245.37
9884188,273.33,268.13,252.77,262.42
70558267,281.99,275.28,253.46,264.07
32872149,256.78,264.53,248.69,258.00
14002835,280.59,273.66,253.05,264.33
...,...,...,...,...
29948179,233.75,240.96,245.74,248.49
27055259,277.70,275.82,260.61,267.44
44495506,264.91,263.27,252.14,260.15
50077389,257.39,252.17,256.54,257.24


In [79]:
y_train = df_ocean_initial_above_threshold_training_data_sampled[rainrate_label]

In [80]:
# Convert labels to a logarithmic scale
y_train_log = np.log(y_train)

In [81]:
# Split data into training and validation sets
X_train, X_val, y_train_log, y_val_log = train_test_split(X_train, y_train_log, test_size = 0.2, random_state = 16)

In [82]:
rf = RandomForestRegressor(n_estimators = 100, random_state = 50)

# Train the model
rf.fit(X_train, y_train_log)

RandomForestRegressor(random_state=50)

In [83]:
# Make predictions on the validation set
y_val_pred_log = rf.predict(X_val)

In [84]:
y_val_pred = np.exp(y_val_pred_log)

In [85]:
y_val = np.exp(y_val_log)

In [86]:
# Calculate R-squared
r2 = r2_score(y_val, y_val_pred)
print("R-squared:", r2)

# Calculate Mean Squared Error
mse = mean_squared_error(y_val, y_val_pred)
rmse = np.sqrt(mse)
print("Root Mean Squared Error:", rmse)

# Calculate Mean Absolute Error
mae = mean_absolute_error(y_val, y_val_pred)
print("Mean Absolute Error:", mae)

correlation_coefficient = pearsonr(y_val, y_val_pred)[0]
print(f'Correlation Coefficient: {correlation_coefficient:.2f}')

R-squared: 0.03945527846018404
Root Mean Squared Error: 7.144803928260839
Mean Absolute Error: 1.180983253877341
Correlation Coefficient: 0.24


In [87]:
indices = classification_prediction_df['Index']

In [88]:
selected_rows = df_ocean_initial.loc[indices]

In [89]:
selected_rows

,ii,jj,iyr,imo,idy,ihr,imn,isc,zlat1,zlon1,...,zDPR3,zlat,zlon,zpcp,ipop,isfc,itemp,itcw,IYNo,IYNo_Radar
26808,244,110,2015,1,1,4,42,13,-55.177,-118.921,...,1.27,-55.177,-118.921,1.01,96,1,280,19,1,1
32472,1377,109,2015,1,1,5,17,37,60.340,-31.919,...,0.25,60.340,-31.919,0.21,29,1,276,6,1,1
39114,98,113,2015,1,1,6,10,11,-64.218,-171.957,...,0.62,-64.218,-171.957,0.66,89,1,275,24,1,1
40124,300,113,2015,1,1,6,16,30,-50.410,-134.736,...,1.93,-50.410,-134.736,0.32,81,1,282,14,1,1
40184,312,113,2015,1,1,6,16,52,-49.301,-133.358,...,3.53,-49.301,-133.358,4.31,100,1,282,16,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73915624,2755,113,2015,12,31,22,41,36,-52.357,69.653,...,0.01,-52.357,69.653,0.96,94,1,275,18,1,0
73915722,2775,111,2015,12,31,22,42,14,-54.008,72.446,...,0.10,-54.008,72.446,0.30,49,1,274,13,1,0
73919914,692,112,2015,12,31,23,9,40,-9.288,-158.433,...,0.00,-9.288,-158.433,0.05,44,1,300,65,0,0
73920023,714,111,2015,12,31,23,10,21,-6.832,-157.501,...,0.00,-6.832,-157.501,0.06,38,1,302,64,1,0


In [90]:
# Run for "ocean_ATMS_HF"
X_test = selected_rows[['ztcs18', 'ztcs21', 'ztcs23', 'ztcs24']]

In [91]:
# Display y_test
X_test

,ztcs18,ztcs21,ztcs23,ztcs24
26808,253.62,246.90,245.47,248.06
32472,244.04,259.28,249.88,257.89
39114,267.18,266.11,253.58,261.45
40124,261.12,267.58,256.54,263.47
40184,261.04,257.44,255.31,259.16
...,...,...,...,...
73915624,266.61,266.25,249.67,260.81
73915722,235.26,237.51,245.52,245.48
73919914,281.64,279.96,256.33,267.71
73920023,283.04,278.32,256.81,269.52


In [92]:
y_test = selected_rows[rainrate_label]

In [93]:
# Display y_test
y_test

26808       1.04
32472       1.57
39114       0.38
40124       1.85
40184       5.13
            ... 
73915624    0.00
73915722    0.00
73919914    0.00
73920023    0.00
73928869    0.00
Name: zDPR1, Length: 6976, dtype: float64

In [94]:
# Make predictions on the test dataset
y_test_pred_log = rf.predict(X_test)

In [95]:
# Applying anti-log over the results
y_test_pred = np.exp(y_test_pred_log)

In [96]:
# Calculate R-squared
r2 = r2_score(y_test, y_test_pred)
print(f'R-squared: {r2:.2f}')

# Calculate Mean Squared Error
mse = mean_squared_error(y_test, y_test_pred)
rmse = np.sqrt(mse)
print(f'Root Mean Squared Error: {rmse:.2f}')

# Calculate Mean Absolute Error
mae = mean_absolute_error(y_test, y_test_pred)
print(f'Mean Absolute Error: {mae:.2f}')

correlation_coefficient = pearsonr(y_test, y_test_pred)[0]
print(f'Correlation Coefficient: {correlation_coefficient:.2f}')

R-squared: 0.07
Root Mean Squared Error: 5.56
Mean Absolute Error: 1.13
Correlation Coefficient: 0.32


## Part 2: Land Data

In [97]:
df_land_initial.loc[:, binary_column_for_rain_norain] = np.where(df_land_initial[rainrate_label] >= rainrate_threshold, 1, 0)

/tmp/ipykernel_121772/2562160300.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_land_initial.loc[:, binary_column_for_rain_norain] = np.where(df_land_initial[rainrate_label] >= rainrate_threshold, 1, 0)


In [98]:
counts = df_land_initial[binary_column_for_rain_norain].value_counts()
count_of_ones = counts[1]

In [99]:
count_of_initial_rows_to_delete = len(df_land_initial) - (2 * counts[1])
count_of_initial_rows_to_delete

7800125

In [100]:
# Step 1: Get all rows where the condition is met
rows_to_delete = df_land_initial[df_land_initial[binary_column_for_rain_norain] == 0]

In [101]:
# Step 2: Remove the first `x` rows from the filtered rows
df_land_initial = df_land_initial.drop(rows_to_delete.iloc[:count_of_initial_rows_to_delete].index)

In [102]:
df_land = df_land_initial.sample(frac = 0.01, random_state = 38)

In [103]:
# Printing the df
df_land

,ii,jj,iyr,imo,idy,ihr,imn,isc,zlat1,zlon1,...,zDPR3,zlat,zlon,zpcp,ipop,isfc,itemp,itcw,IYNo,IYNo_Radar
31820709,1644,113,2015,5,25,6,9,2,60.952,112.254,...,1.03,60.952,112.254,0.31,51,3,286,14,1,1
73314122,2403,110,2015,12,28,23,33,42,-23.924,44.732,...,0.00,-23.924,44.732,0.01,2,4,297,40,0,0
53468203,1648,110,2015,8,31,3,42,59,60.916,80.124,...,0.68,60.916,80.124,1.28,95,3,283,28,1,1
71843607,2273,111,2015,12,22,8,37,7,-9.594,-69.366,...,5.59,-9.594,-69.366,6.41,100,3,296,59,1,1
71550389,2010,110,2015,12,21,1,37,46,19.462,27.152,...,0.00,19.462,27.152,0.00,0,7,279,8,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72440919,2074,110,2015,12,24,23,45,32,12.376,41.595,...,0.00,12.376,41.595,0.01,3,5,298,42,0,0
70937725,2190,110,2015,12,18,8,55,51,-0.319,-60.960,...,0.00,-0.319,-60.960,0.03,5,3,298,52,0,0
8619044,583,111,2015,2,9,14,21,4,-21.251,-45.001,...,0.05,-21.251,-45.001,0.02,3,4,295,36,0,1
72691078,2476,110,2015,12,26,2,11,26,-31.774,22.794,...,0.00,-31.774,22.794,0.00,0,5,282,8,0,0


### 2.1.1) Land Data, "land_ATMS_LF+HF", Random Forest, Step 1, Classification (Rain/No-Rain)

In [104]:
# Run for "land_ATMS_LF+HF"
X = df_land[['ztcs15', 'ztcs16', 'ztcs18', 'ztcs21', 'ztcs23', 'ztcs24']]

In [105]:
# Printing the df
X

,ztcs15,ztcs16,ztcs18,ztcs21,ztcs23,ztcs24
31820709,274.29,274.18,269.05,242.92,243.74,243.03
73314122,285.68,285.04,286.66,280.84,257.18,268.89
53468203,275.74,275.21,264.85,258.27,247.82,256.17
71843607,271.54,261.57,214.97,146.73,190.36,159.28
71550389,274.11,271.47,264.51,267.97,265.79,273.66
...,...,...,...,...,...,...
72440919,283.37,281.31,284.68,279.76,261.24,269.77
70937725,284.90,283.23,286.39,278.54,255.61,267.79
8619044,283.29,282.27,283.22,273.58,251.43,261.68
72691078,276.05,274.01,272.38,277.44,273.59,278.25


In [106]:
y = df_land[binary_column_for_rain_norain]

In [107]:
# Split data into training and validation sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 16)

In [108]:
# Measure how much time it takes to read in allvalues from all 3 columns
start_time = time.time()

rf = RandomForestClassifier(n_estimators = 200, random_state = 50)

# Train the model
rf.fit(X_train, y_train)

# Mark the end of code execution
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Execution time: {elapsed_time:.5f} seconds")

Execution time: 1.38888 seconds


In [109]:
# Make predictions on the test set
y_test_pred = rf.predict(X_test)

In [110]:
# Calculating metrics
accuracy = accuracy_score(y_test, y_test_pred)
conf_matrix = confusion_matrix(y_test, y_test_pred)
class_report = classification_report(y_test, y_test_pred)

In [111]:
# Expressing the confusion matrix in terms of ratios (normalized)
conf_matrix_normalized = np.round(conf_matrix.astype('float') / conf_matrix.sum(axis=1)[:, np.newaxis] * 100, 2)

In [112]:
# Print evaluation metrics
print(f'Accuracy: {accuracy * 100:.2f}%')
print('\nArrangement of the confusion matrix below: \n [[TN \t FP] \n [FN \t TP]]')
print('\nConfusion Matrix:')
print(conf_matrix_normalized)
print('\nClassification Report:')
print(class_report)

Accuracy: 90.64%

Arrangement of the confusion matrix below: 
 [[TN 	 FP] 
 [FN 	 TP]]

Confusion Matrix:
[[89.74 10.26]
 [ 8.42 91.58]]

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.90      0.91       692
           1       0.90      0.92      0.91       665

    accuracy                           0.91      1357
   macro avg       0.91      0.91      0.91      1357
weighted avg       0.91      0.91      0.91      1357



In [113]:
# Getting indices of X_val
index_numbers = X_test.index.tolist()

In [114]:
# Create the DataFrame with only entries where Predictions is 1
classification_prediction_df = pd.DataFrame({
    'Index': [index for i, index in enumerate(index_numbers) if y_test_pred[i] == 1],
    'Predictions': [pred for pred in y_test_pred if pred == 1]
})

In [115]:
# classification_prediction_df = classification_prediction_df.sort_values(by = 'Index').reset_index(drop = False)
classification_prediction_df = classification_prediction_df.sort_values(by = 'Index').reset_index(drop = True)

In [116]:
# # Printing the df
# classification_prediction_df

### 2.1.2) Land Data, "land_ATMS_LF+HF", Random Forest, Step 2, Regression (Amount of Rain)

In [117]:
df_land_initial_above_threshold = df_land_initial[(df_land_initial[rainrate_label] >= rainrate_threshold)]

In [118]:
df_land_initial_above_threshold

,ii,jj,iyr,imo,idy,ihr,imn,isc,zlat1,zlon1,...,zDPR3,zlat,zlon,zpcp,ipop,isfc,itemp,itcw,IYNo,IYNo_Radar
2470,577,109,2015,1,1,1,47,33,-21.824,-43.634,...,1.48,-21.824,-43.634,1.55,97,3,295,42,1,1
2471,577,110,2015,1,1,1,47,33,-21.848,-43.585,...,1.67,-21.848,-43.585,1.63,97,3,295,42,1,1
2472,577,111,2015,1,1,1,47,33,-21.872,-43.536,...,1.61,-21.872,-43.536,1.59,97,3,295,42,1,1
2505,584,109,2015,1,1,1,47,46,-21.061,-43.273,...,2.18,-21.061,-43.273,5.21,100,3,296,39,1,1
2506,584,110,2015,1,1,1,47,46,-21.085,-43.224,...,2.06,-21.085,-43.224,3.03,97,3,296,39,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73927251,2515,109,2016,1,1,0,6,38,-28.308,26.029,...,0.20,-28.308,26.029,0.12,25,5,295,32,1,1
73927287,2522,110,2016,1,1,0,6,51,-29.078,26.387,...,0.06,-29.078,26.387,0.10,24,5,295,31,0,1
73927288,2522,111,2016,1,1,0,6,51,-29.103,26.335,...,0.06,-29.103,26.335,0.45,67,5,295,31,1,1
73927356,2536,109,2016,1,1,0,7,18,-30.540,27.281,...,1.54,-30.540,27.281,0.68,71,4,292,28,1,1


In [119]:
# Filter out rows where the index matches indices_to_delete
df_land_initial_above_threshold_training_data = df_land_initial_above_threshold.drop(index = classification_prediction_df['Index'][classification_prediction_df['Index'].isin(df_land_initial_above_threshold.index)])
# df_land_initial_above_threshold_training_data

In [120]:
df_land_initial_above_threshold_training_data_sampled = df_land_initial_above_threshold_training_data.sample(frac = 0.05, random_state = 38)
df_land_initial_above_threshold_training_data_sampled

,ii,jj,iyr,imo,idy,ihr,imn,isc,zlat1,zlon1,...,zDPR3,zlat,zlon,zpcp,ipop,isfc,itemp,itcw,IYNo,IYNo_Radar
24755933,1730,113,2015,4,22,23,52,40,48.432,13.561,...,1.61,48.432,13.561,0.35,42,4,282,13,1,1
7552520,2543,113,2015,2,4,7,21,57,-38.731,-64.657,...,5.93,-38.731,-64.657,6.04,100,5,294,39,1,1
47205070,1353,109,2015,8,3,6,3,10,58.634,92.801,...,1.15,58.634,92.801,0.78,85,3,295,27,1,1
69469756,537,110,2015,12,11,21,47,25,-18.654,-54.528,...,2.67,-18.654,-54.528,0.03,5,3,300,49,0,1
16340015,1081,113,2015,3,15,22,22,9,33.239,72.313,...,1.24,33.239,72.313,1.47,96,5,285,25,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47045684,1620,109,2015,8,2,13,13,17,62.480,56.973,...,0.94,62.480,56.973,0.47,71,3,288,25,1,1
73621727,646,111,2015,12,30,11,39,48,-6.628,21.725,...,1.23,-6.628,21.725,1.36,93,3,302,46,1,1
46667921,2068,110,2015,7,31,21,20,29,20.888,-5.417,...,0.18,20.888,-5.417,0.12,22,7,311,42,1,1
60990300,2527,111,2015,11,3,9,21,18,-29.581,145.220,...,0.96,-29.581,145.220,1.55,92,5,297,43,1,1


In [121]:
# Run for "land_ATMS_LF+HF"
X_train = df_land_initial_above_threshold_training_data_sampled[['ztcs15', 'ztcs16', 'ztcs18', 'ztcs21', 'ztcs23', 'ztcs24']]

In [122]:
X_train

,ztcs15,ztcs16,ztcs18,ztcs21,ztcs23,ztcs24
24755933,266.82,264.43,263.79,261.14,249.15,254.13
7552520,272.08,259.05,201.26,158.02,188.74,167.15
47205070,279.17,275.57,269.50,266.85,252.14,258.41
69469756,286.52,283.73,286.47,279.33,261.04,269.24
16340015,266.29,266.42,258.36,240.18,246.72,245.68
...,...,...,...,...,...,...
47045684,276.13,276.06,270.56,267.83,251.75,261.38
73621727,283.39,279.53,257.34,241.29,252.31,251.29
46667921,297.23,295.10,286.51,277.10,255.47,264.65
60990300,286.19,278.78,246.50,236.48,248.94,248.18


In [123]:
y_train = df_land_initial_above_threshold_training_data_sampled[rainrate_label]

In [124]:
# Convert labels to a logarithmic scale
y_train_log = np.log(y_train)

In [125]:
# Split data into training and validation sets
X_train, X_val, y_train_log, y_val_log = train_test_split(X_train, y_train_log, test_size = 0.2, random_state = 16)

In [126]:
rf = RandomForestRegressor(n_estimators = 100, random_state = 50)

In [127]:
# Train the model
rf.fit(X_train, y_train_log)

RandomForestRegressor(random_state=50)

In [128]:
# Make predictions on the validation set
y_val_pred_log = rf.predict(X_val)

In [129]:
y_val_pred = np.exp(y_val_pred_log)

In [130]:
y_val = np.exp(y_val_log)

In [131]:
# Calculate R-squared
r2 = r2_score(y_val, y_val_pred)
print("R-squared:", r2)

# Calculate Mean Squared Error
mse = mean_squared_error(y_val, y_val_pred)
rmse = np.sqrt(mse)
print("Root Mean Squared Error:", rmse)

# Calculate Mean Absolute Error
mae = mean_absolute_error(y_val, y_val_pred)
print("Mean Absolute Error:", mae)

correlation_coefficient = pearsonr(y_val, y_val_pred)[0]
print(f'Correlation Coefficient: {correlation_coefficient:.2f}')

R-squared: 0.09310375371683255
Root Mean Squared Error: 5.969683796489398
Mean Absolute Error: 1.3489346575786891
Correlation Coefficient: 0.36


In [132]:
indices = classification_prediction_df['Index']

In [133]:
selected_rows = df_land_initial.loc[indices]
selected_rows

,ii,jj,iyr,imo,idy,ihr,imn,isc,zlat1,zlon1,...,zDPR3,zlat,zlon,zpcp,ipop,isfc,itemp,itcw,IYNo,IYNo_Radar
224496,741,112,2015,1,2,4,5,42,-3.856,-73.897,...,21.98,-3.856,-73.897,2.72,99,3,296,55,1,1
636931,1027,113,2015,1,4,5,35,31,35.090,-84.823,...,3.39,35.090,-84.823,3.80,100,4,288,37,1,1
699523,563,113,2015,1,4,13,3,39,-15.697,135.328,...,3.92,-15.697,135.328,0.30,38,4,299,62,1,1
1075697,2209,112,2015,1,6,9,5,41,-2.603,15.626,...,2.05,-2.603,15.626,1.44,92,3,299,47,1,1
1525968,418,109,2015,1,8,11,3,22,-31.407,139.515,...,0.33,-31.407,139.515,0.48,56,6,298,41,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73750949,428,112,2015,12,31,2,58,24,-30.294,136.586,...,0.61,-30.294,136.586,0.73,87,6,302,31,1,1
73773673,2365,110,2015,12,31,5,31,28,-19.734,-56.865,...,0.00,-19.734,-56.865,0.09,13,3,299,58,0,0
73862758,2452,109,2015,12,31,16,21,58,-21.532,139.686,...,0.00,-21.532,139.686,0.12,27,6,296,44,1,0
73868727,721,113,2015,12,31,17,0,25,-6.101,-63.429,...,3.89,-6.101,-63.429,1.60,84,3,306,58,1,1


In [134]:
# Run for "land_ATMS_LF+HF"
X_test = selected_rows[['ztcs15', 'ztcs16', 'ztcs18', 'ztcs21', 'ztcs23', 'ztcs24']]

In [135]:
X_test

,ztcs15,ztcs16,ztcs18,ztcs21,ztcs23,ztcs24
224496,278.94,272.11,238.14,201.37,195.83,190.65
636931,271.89,266.57,256.84,257.12,250.87,260.20
699523,285.90,284.75,280.95,270.21,257.03,263.90
1075697,280.31,277.17,262.61,243.54,248.57,247.85
1525968,289.36,285.41,274.75,267.56,258.00,264.86
...,...,...,...,...,...,...
73750949,290.41,284.87,262.81,248.74,249.54,251.22
73773673,284.58,281.33,284.15,274.24,255.20,264.07
73862758,278.61,276.69,278.93,269.65,261.70,264.61
73868727,281.33,274.08,262.10,225.90,221.31,220.65


In [136]:
y_test = selected_rows[rainrate_label]

In [137]:
y_test

224496       7.49
636931       1.54
699523       0.40
1075697      3.58
1525968      0.28
            ...  
73750949     0.41
73773673     0.00
73862758     0.00
73868727    17.04
73897358     0.89
Name: zDPR1, Length: 680, dtype: float64

In [138]:
# Make predictions on the test dataset
y_test_pred_log = rf.predict(X_test)

In [139]:
y_test_pred_log

array([ 6.14369440e-01,  1.24545752e+00, -7.34633276e-01, -7.14423297e-01,
       -7.44784698e-01, -3.38365085e-01, -9.64423903e-01, -3.50634612e-01,
        1.88987669e-01, -1.09090750e+00,  8.66695756e-02, -6.25840667e-01,
        9.23754040e-01, -4.64440513e-01, -2.87343151e-01, -6.27780610e-01,
        1.95154397e+00,  2.10536049e+00, -7.81816105e-01,  1.57662746e-01,
       -9.15697710e-01, -1.05263344e+00, -4.11272730e-01,  1.17500966e+00,
       -3.59280732e-01, -5.13034585e-01, -8.21334347e-01, -2.45063042e-01,
       -8.91824564e-01, -6.13458870e-01, -7.60900782e-01,  1.50174745e-01,
        2.18266521e+00,  1.59956164e+00,  3.36510009e-02,  1.65567084e+00,
       -6.92176508e-01,  7.27057785e-01,  1.74703298e-01,  1.65956993e-01,
       -1.80867481e-01, -9.43076265e-01, -1.09778116e+00, -5.99768141e-01,
        2.07431485e-03, -1.16352686e+00, -4.52107987e-01,  3.51158432e-01,
       -4.74596932e-01, -5.23329139e-01,  1.31349498e+00, -4.99567666e-01,
        6.99844581e-01,  

In [140]:
# Applying anti-log over the results
y_test_pred = np.exp(y_test_pred_log)

In [141]:
# Calculate R-squared
r2 = r2_score(y_test, y_test_pred)
print(f'R-squared: {r2:.2f}')

# Calculate Mean Squared Error
mse = mean_squared_error(y_test, y_test_pred)
rmse = np.sqrt(mse)
print(f'Root Mean Squared Error: {rmse:.2f}')

# Calculate Mean Absolute Error
mae = mean_absolute_error(y_test, y_test_pred)
print(f'Mean Absolute Error: {mae:.2f}')

correlation_coefficient = pearsonr(y_test, y_test_pred)[0]
print(f'Correlation Coefficient: {correlation_coefficient:.2f}')

R-squared: 0.08
Root Mean Squared Error: 12.32
Mean Absolute Error: 1.90
Correlation Coefficient: 0.46


### 2.2.1) Land Data, "land_ATMS_HF", Random Forest, Step 1, Classification (Rain/No-Rain)

In [142]:
# Run for "land_ATMS_HF"
X = df_land[['ztcs18', 'ztcs21', 'ztcs23', 'ztcs24']]

In [143]:
# # Printing the df
# X

In [144]:
y = df_land[binary_column_for_rain_norain]

In [145]:
# Split data into training and validation sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 16)

In [146]:
# Measure how much time it takes to read in allvalues from all 3 columns
start_time = time.time()

rf = RandomForestClassifier(n_estimators = 200, random_state = 50)

# Train the model
rf.fit(X_train, y_train)

# Mark the end of code execution
end_time = time.time()
elapsed_time = end_time - start_time
print(f"Execution time: {elapsed_time:.5f} seconds")

Execution time: 1.49099 seconds


In [147]:
# Make predictions on the test set
y_test_pred = rf.predict(X_test)

In [148]:
# Calculating metrics
accuracy = accuracy_score(y_test, y_test_pred)
conf_matrix = confusion_matrix(y_test, y_test_pred)
class_report = classification_report(y_test, y_test_pred)

In [149]:
# Expressing the confusion matrix in terms of ratios (normalized)
conf_matrix_normalized = np.round(conf_matrix.astype('float') / conf_matrix.sum(axis=1)[:, np.newaxis] * 100, 2)

In [150]:
# Print evaluation metrics
print(f'Accuracy: {accuracy * 100:.2f}%')
print('\nArrangement of the confusion matrix below: \n [[TN \t FP] \n [FN \t TP]]')
print('\nConfusion Matrix:')
print(conf_matrix_normalized)
print('\nClassification Report:')
print(class_report)

Accuracy: 88.06%

Arrangement of the confusion matrix below: 
 [[TN 	 FP] 
 [FN 	 TP]]

Confusion Matrix:
[[87.14 12.86]
 [10.98 89.02]]

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.87      0.88       692
           1       0.87      0.89      0.88       665

    accuracy                           0.88      1357
   macro avg       0.88      0.88      0.88      1357
weighted avg       0.88      0.88      0.88      1357



In [151]:
# Getting indices of X_val
index_numbers = X_test.index.tolist()

In [152]:
# Create the DataFrame with only entries where Predictions is 1
classification_prediction_df = pd.DataFrame({
    'Index': [index for i, index in enumerate(index_numbers) if y_test_pred[i] == 1],
    'Predictions': [pred for pred in y_test_pred if pred == 1]
})

In [153]:
# classification_prediction_df = classification_prediction_df.sort_values(by = 'Index').reset_index(drop = False)
classification_prediction_df = classification_prediction_df.sort_values(by = 'Index').reset_index(drop = True)

In [154]:
# Printing the df
classification_prediction_df

,Index,Predictions
0,224496,1
1,699523,1
2,1075697,1
3,1525968,1
4,1548129,1
...,...,...
676,73862758,1
677,73868727,1
678,73897358,1
679,73897970,1


### 2.2.2) Land Data, "land_ATMS_HF", Random Forest, Step 2, Regression (Amount of Rain)

In [155]:
df_land_initial_above_threshold = df_land_initial[(df_land_initial[rainrate_label] >= rainrate_threshold)]

In [156]:
# Filter out rows where the index matches indices_to_delete
df_land_initial_above_threshold_training_data = df_land_initial_above_threshold.drop(index = classification_prediction_df['Index'][classification_prediction_df['Index'].isin(df_land_initial_above_threshold.index)])
# df_land_initial_above_threshold_training_data

In [157]:
df_land_initial_above_threshold_training_data_sampled = df_land_initial_above_threshold_training_data.sample(frac = 0.05, random_state = 38)
df_land_initial_above_threshold_training_data_sampled

,ii,jj,iyr,imo,idy,ihr,imn,isc,zlat1,zlon1,...,zDPR3,zlat,zlon,zpcp,ipop,isfc,itemp,itcw,IYNo,IYNo_Radar
3121486,2380,109,2015,1,15,12,13,34,-21.424,-62.313,...,0.14,-21.424,-62.313,0.40,59,3,296,55,1,1
62228124,2340,113,2015,11,8,20,22,49,-9.336,-55.112,...,0.38,-9.336,-55.112,0.02,3,3,301,45,0,1
31120873,1235,111,2015,5,22,3,53,6,48.718,68.629,...,0.73,48.718,68.629,0.28,30,5,283,16,1,1
5195168,1181,112,2015,1,24,17,41,51,50.365,18.672,...,0.43,50.365,18.672,0.02,6,4,272,12,0,1
6625409,2295,111,2015,1,31,1,25,41,-12.029,26.732,...,0.17,-12.029,26.732,0.82,87,3,291,36,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36064964,1389,110,2015,6,14,1,13,52,64.249,50.554,...,0.52,64.249,50.554,0.43,66,3,280,19,1,1
29232652,2171,109,2015,5,13,19,47,27,9.616,10.369,...,6.50,9.616,10.369,1.14,89,4,303,51,1,1
49465558,825,113,2015,8,13,4,53,41,5.444,20.027,...,1.50,5.444,20.027,2.81,99,3,296,52,1,1
14681428,2222,110,2015,3,8,15,6,40,3.833,13.676,...,0.51,3.833,13.676,0.40,26,3,306,40,1,1


In [158]:
# Run for "land_ATMS_HF"
X_train = df_land_initial_above_threshold_training_data_sampled[['ztcs18', 'ztcs21', 'ztcs23', 'ztcs24']]
X_train

,ztcs18,ztcs21,ztcs23,ztcs24
3121486,274.72,255.06,249.77,256.07
62228124,287.52,278.89,254.62,267.06
31120873,266.44,265.25,249.27,258.35
5195168,260.34,260.47,249.01,257.15
6625409,268.85,253.44,247.66,251.14
...,...,...,...,...
36064964,266.99,260.24,249.42,258.13
29232652,226.75,173.82,193.10,177.51
49465558,248.86,226.67,247.74,240.15
14681428,284.73,266.61,256.95,262.76


In [159]:
y_train = df_land_initial_above_threshold_training_data_sampled[rainrate_label]

In [160]:
# Convert labels to a logarithmic scale
y_train_log = np.log(y_train)

In [161]:
# Split data into training and validation sets
X_train, X_val, y_train_log, y_val_log = train_test_split(X_train, y_train_log, test_size = 0.2, random_state = 16)

In [162]:
rf = RandomForestRegressor(n_estimators = 100, random_state = 50)

# Train the model
rf.fit(X_train, y_train_log)

RandomForestRegressor(random_state=50)

In [163]:
# Make predictions on the validation set
y_val_pred_log = rf.predict(X_val)

In [164]:
y_val_pred = np.exp(y_val_pred_log)

In [165]:
y_val = np.exp(y_val_log)

In [166]:
# Calculate R-squared
r2 = r2_score(y_val, y_val_pred)
print("R-squared:", r2)

# Calculate Mean Squared Error
mse = mean_squared_error(y_val, y_val_pred)
rmse = np.sqrt(mse)
print("Root Mean Squared Error:", rmse)

# Calculate Mean Absolute Error
mae = mean_absolute_error(y_val, y_val_pred)
print("Mean Absolute Error:", mae)

correlation_coefficient = pearsonr(y_val, y_val_pred)[0]
print(f'Correlation Coefficient: {correlation_coefficient:.2f}')

R-squared: 0.048591844265123196
Root Mean Squared Error: 5.804246095229063
Mean Absolute Error: 1.5536730951712323
Correlation Coefficient: 0.28


In [167]:
indices = classification_prediction_df['Index']

In [168]:
selected_rows = df_land_initial.loc[indices]
selected_rows

,ii,jj,iyr,imo,idy,ihr,imn,isc,zlat1,zlon1,...,zDPR3,zlat,zlon,zpcp,ipop,isfc,itemp,itcw,IYNo,IYNo_Radar
224496,741,112,2015,1,2,4,5,42,-3.856,-73.897,...,21.98,-3.856,-73.897,2.72,99,3,296,55,1,1
699523,563,113,2015,1,4,13,3,39,-15.697,135.328,...,3.92,-15.697,135.328,0.30,38,4,299,62,1,1
1075697,2209,112,2015,1,6,9,5,41,-2.603,15.626,...,2.05,-2.603,15.626,1.44,92,3,299,47,1,1
1525968,418,109,2015,1,8,11,3,22,-31.407,139.515,...,0.33,-31.407,139.515,0.48,56,6,298,41,1,1
1548129,2204,110,2015,1,8,13,31,45,-1.930,-60.902,...,0.11,-1.930,-60.902,1.13,75,3,300,54,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73862758,2452,109,2015,12,31,16,21,58,-21.532,139.686,...,0.00,-21.532,139.686,0.12,27,6,296,44,1,0
73868727,721,113,2015,12,31,17,0,25,-6.101,-63.429,...,3.89,-6.101,-63.429,1.60,84,3,306,58,1,1
73897358,1663,113,2015,12,31,20,34,57,59.724,12.082,...,0.36,59.724,12.082,0.60,73,5,273,13,1,1
73897970,1788,109,2015,12,31,20,38,51,49.704,31.102,...,0.00,49.704,31.102,0.00,1,4,265,4,0,0


In [169]:
# Run for "land_ATMS_HF"
X_test = selected_rows[['ztcs18', 'ztcs21', 'ztcs23', 'ztcs24']]

# Display y_test
X_test

,ztcs18,ztcs21,ztcs23,ztcs24
224496,238.14,201.37,195.83,190.65
699523,280.95,270.21,257.03,263.90
1075697,262.61,243.54,248.57,247.85
1525968,274.75,267.56,258.00,264.86
1548129,276.72,270.64,258.67,266.35
...,...,...,...,...
73862758,278.93,269.65,261.70,264.61
73868727,262.10,225.90,221.31,220.65
73897358,253.49,243.32,247.26,247.40
73897970,257.82,255.50,250.43,255.43


In [170]:
y_test = selected_rows[rainrate_label]

# Display y_test
y_test

224496       7.49
699523       0.40
1075697      3.58
1525968      0.28
1548129      0.23
            ...  
73862758     0.00
73868727    17.04
73897358     0.89
73897970     0.00
73926480     0.00
Name: zDPR1, Length: 681, dtype: float64

In [171]:
# Make predictions on the test dataset
y_test_pred_log = rf.predict(X_test)

In [172]:
# Applying anti-log over the results
y_test_pred = np.exp(y_test_pred_log)

In [173]:
# Calculate R-squared
r2 = r2_score(y_test, y_test_pred)
print(f'R-squared: {r2:.2f}')

# Calculate Mean Squared Error
mse = mean_squared_error(y_test, y_test_pred)
rmse = np.sqrt(mse)
print(f'Root Mean Squared Error: {rmse:.2f}')

# Calculate Mean Absolute Error
mae = mean_absolute_error(y_test, y_test_pred)
print(f'Mean Absolute Error: {mae:.2f}')

correlation_coefficient = pearsonr(y_test, y_test_pred)[0]
print(f'Correlation Coefficient: {correlation_coefficient:.2f}')

R-squared: 0.04
Root Mean Squared Error: 12.49
Mean Absolute Error: 1.99
Correlation Coefficient: 0.31
